In [1]:
"""
Double DQN for the Inverted Pendulum (Gymnasium's Pendulum-v1).

Pendulum-v1 has a *continuous* 1-D torque action, so we discretize it into
N_ACTIONS evenly spaced bins and let a standard Double DQN pick among them.

Pipeline (matches the mental model from the design doc):

    Environment  -->  Agent (DQN)  -->  Replay Buffer  -->  Training Loop
          ^_____________________________________________________|

Run:
    python dqn_pendulum.py
"""

import random
from collections import deque, namedtuple

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import gymnasium as gym
import matplotlib.pyplot as plt

# ----------------------------------------------------------------------
# Hyperparameters
# ----------------------------------------------------------------------
ENV_NAME = "Pendulum-v1"

N_ACTIONS = 21              # odd -> exact zero-torque bin is representable
TORQUE_MIN, TORQUE_MAX = -2.0, 2.0

GAMMA = 0.99
LEARNING_RATE = 1e-3
BATCH_SIZE = 64
GRAD_CLIP_NORM = 1.0

EPISODES = 300
MAX_STEPS_PER_EPISODE = 200

EPSILON_START = 1.0
EPSILON_END = 0.05
# Recalibrated so epsilon actually reaches EPSILON_END by the last episode:
#   decay = (end / start) ** (1 / episodes)
EPSILON_DECAY = (EPSILON_END / EPSILON_START) ** (1.0 / EPISODES)

BUFFER_SIZE = 50_000
START_TRAINING_AFTER = 1_000   # env steps, not episodes

TAU = 0.005                 # soft target update rate (Polyak averaging)

SEED = 0
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ----------------------------------------------------------------------
# Action discretization: index -> torque
# ----------------------------------------------------------------------
TORQUE_VALUES = np.linspace(TORQUE_MIN, TORQUE_MAX, N_ACTIONS)  # shape (N_ACTIONS,)


def action_index_to_torque(action_index: int) -> np.ndarray:
    """Map a discrete action index to the continuous torque Pendulum-v1 expects."""
    return np.array([TORQUE_VALUES[action_index]], dtype=np.float32)


# ----------------------------------------------------------------------
# Q-network: small, since the task is low-dimensional
# ----------------------------------------------------------------------
class QNetwork(nn.Module):
    def __init__(self, state_dim: int, n_actions: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, n_actions),
        )

    def forward(self, x):
        return self.net(x)


# ----------------------------------------------------------------------
# Replay buffer
# ----------------------------------------------------------------------
Transition = namedtuple("Transition", ["state", "action", "reward", "next_state", "done"])


class ReplayBuffer:
    def __init__(self, capacity: int):
        self.buffer = deque(maxlen=capacity)

    def add(self, state, action, reward, next_state, done):
        self.buffer.append(Transition(state, action, reward, next_state, done))

    def sample(self, batch_size: int):
        batch = random.sample(self.buffer, batch_size)
        states = torch.tensor(np.array([t.state for t in batch]), dtype=torch.float32)
        actions = torch.tensor([t.action for t in batch], dtype=torch.long).unsqueeze(1)
        rewards = torch.tensor([t.reward for t in batch], dtype=torch.float32).unsqueeze(1)
        next_states = torch.tensor(np.array([t.next_state for t in batch]), dtype=torch.float32)
        dones = torch.tensor([t.done for t in batch], dtype=torch.float32).unsqueeze(1)
        return states, actions, rewards, next_states, dones

    def __len__(self):
        return len(self.buffer)


# ----------------------------------------------------------------------
# Agent: action selection + learning step + soft target update
# ----------------------------------------------------------------------
class DQNAgent:
    def __init__(self, state_dim: int, n_actions: int):
        self.q_online = QNetwork(state_dim, n_actions).to(DEVICE)
        self.q_target = QNetwork(state_dim, n_actions).to(DEVICE)
        self.q_target.load_state_dict(self.q_online.state_dict())  # start identical

        self.optimizer = optim.Adam(self.q_online.parameters(), lr=LEARNING_RATE)
        self.loss_fn = nn.MSELoss()  # swap for nn.SmoothL1Loss() later to compare vs Huber

        self.n_actions = n_actions
        self.epsilon = EPSILON_START

    def select_action(self, state: np.ndarray) -> int:
        if random.random() < self.epsilon:
            return random.randrange(self.n_actions)
        with torch.no_grad():
            state_t = torch.tensor(state, dtype=torch.float32, device=DEVICE).unsqueeze(0)
            q_values = self.q_online(state_t)
            return int(torch.argmax(q_values, dim=1).item())

    def decay_epsilon(self):
        self.epsilon = max(EPSILON_END, self.epsilon * EPSILON_DECAY)

    def learn(self, batch):
        states, actions, rewards, next_states, dones = batch
        states = states.to(DEVICE)
        actions = actions.to(DEVICE)
        rewards = rewards.to(DEVICE)
        next_states = next_states.to(DEVICE)
        dones = dones.to(DEVICE)

        # --- Double DQN target ---
        with torch.no_grad():
            next_actions = torch.argmax(self.q_online(next_states), dim=1, keepdim=True)
            next_q_target = self.q_target(next_states).gather(1, next_actions)
            targets = rewards + GAMMA * next_q_target * (1.0 - dones)

        # --- current Q estimate for the taken actions only ---
        current_q = self.q_online(states).gather(1, actions)

        loss = self.loss_fn(current_q, targets)

        self.optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(self.q_online.parameters(), GRAD_CLIP_NORM)
        self.optimizer.step()

        self._soft_update_target()

        return loss.item()

    def _soft_update_target(self):
        # theta_target <- tau * theta_online + (1 - tau) * theta_target, every learning step
        for target_param, online_param in zip(self.q_target.parameters(), self.q_online.parameters()):
            target_param.data.copy_(TAU * online_param.data + (1.0 - TAU) * target_param.data)


# ----------------------------------------------------------------------
# Training loop
# ----------------------------------------------------------------------
def train():
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)

    env = gym.make(ENV_NAME)
    state_dim = env.observation_space.shape[0]  # 3: [cos(theta), sin(theta), theta_dot]

    agent = DQNAgent(state_dim, N_ACTIONS)
    buffer = ReplayBuffer(BUFFER_SIZE)

    total_steps = 0
    episode_returns = []
    episode_losses = []  # mean training loss per episode (None while buffer is still warming up)

    for episode in range(1, EPISODES + 1):
        state, _ = env.reset(seed=SEED + episode)
        episode_return = 0.0
        step_losses = []

        for _ in range(MAX_STEPS_PER_EPISODE):
            action_index = agent.select_action(state)
            torque = action_index_to_torque(action_index)

            next_state, reward, terminated, truncated, _ = env.step(torque)
            done = terminated or truncated

            buffer.add(state, action_index, reward, next_state, float(done))

            state = next_state
            episode_return += reward
            total_steps += 1

            if len(buffer) >= START_TRAINING_AFTER:
                batch = buffer.sample(BATCH_SIZE)
                loss = agent.learn(batch)
                step_losses.append(loss)

            if done:
                break

        agent.decay_epsilon()
        episode_returns.append(episode_return)
        episode_losses.append(np.mean(step_losses) if step_losses else None)

        if episode % 10 == 0:
            avg_return = np.mean(episode_returns[-10:])
            print(
                f"Episode {episode:4d} | "
                f"return {episode_return:8.1f} | "
                f"avg(last 10) {avg_return:8.1f} | "
                f"epsilon {agent.epsilon:.3f}"
            )

    env.close()
    return agent, episode_returns, episode_losses


def plot_training_curves(episode_returns, episode_losses, save_path="training_curves.png"):
    """Plot episode return (with moving average) and training loss side by side."""
    episodes = np.arange(1, len(episode_returns) + 1)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

    # --- Return curve ---
    axes[0].plot(episodes, episode_returns, alpha=0.35, color="tab:blue", label="Episode return")
    window = 10
    if len(episode_returns) >= window:
        moving_avg = np.convolve(episode_returns, np.ones(window) / window, mode="valid")
        axes[0].plot(
            episodes[window - 1:], moving_avg, color="tab:blue", linewidth=2,
            label=f"{window}-episode moving avg",
        )
    axes[0].set_xlabel("Episode")
    axes[0].set_ylabel("Return")
    axes[0].set_title("Episode Return")
    axes[0].legend()
    axes[0].grid(alpha=0.3)

    # --- Loss curve (skip episodes before training started) ---
    valid = [(e, l) for e, l in zip(episodes, episode_losses) if l is not None]
    if valid:
        loss_episodes, loss_values = zip(*valid)
        axes[1].plot(loss_episodes, loss_values, color="tab:red")
    axes[1].set_xlabel("Episode")
    axes[1].set_ylabel("Mean MSE loss")
    axes[1].set_title("Training Loss")
    axes[1].set_yscale("log")
    axes[1].grid(alpha=0.3)

    fig.tight_layout()
    fig.savefig(save_path, dpi=150)
    plt.close(fig)
    print(f"Saved training curves to {save_path}")


if __name__ == "__main__":
    trained_agent, returns, losses = train()

    # Save the trained online network so it can be reloaded later (e.g. by evaluate.py)
    torch.save(trained_agent.q_online.state_dict(), "dqn_pendulum_weights.pt")
    print("\nSaved trained weights to dqn_pendulum_weights.pt")

    plot_training_curves(returns, losses)

Episode   10 | return  -1680.8 | avg(last 10)  -1304.8 | epsilon 0.905
Episode   20 | return   -995.3 | avg(last 10)  -1140.1 | epsilon 0.819
Episode   30 | return  -1018.5 | avg(last 10)  -1078.0 | epsilon 0.741
Episode   40 | return   -863.0 | avg(last 10)   -976.8 | epsilon 0.671
Episode   50 | return   -626.3 | avg(last 10)   -866.9 | epsilon 0.607
Episode   60 | return   -585.6 | avg(last 10)   -731.0 | epsilon 0.549
Episode   70 | return   -380.1 | avg(last 10)   -512.0 | epsilon 0.497
Episode   80 | return   -401.1 | avg(last 10)   -413.5 | epsilon 0.450
Episode   90 | return   -377.6 | avg(last 10)   -434.8 | epsilon 0.407
Episode  100 | return   -233.6 | avg(last 10)   -345.5 | epsilon 0.368
Episode  110 | return   -597.5 | avg(last 10)   -349.6 | epsilon 0.333
Episode  120 | return   -366.3 | avg(last 10)   -327.5 | epsilon 0.302
Episode  130 | return   -253.1 | avg(last 10)   -258.7 | epsilon 0.273
Episode  140 | return   -122.2 | avg(last 10)   -254.9 | epsilon 0.247
Episod

In [6]:
from google.colab import files
uploaded = files.upload()



Saving dqn_pendulum_weights.pt to dqn_pendulum_weights.pt


In [7]:
"""
Evaluate a trained DQN policy on Pendulum-v1.

Standalone version: does NOT import from dqn_pendulum.py. The few pieces it
needs (QNetwork architecture, constants, torque mapping) are duplicated below.
This is intentional so the file works whether you run it separately or paste
it directly underneath the training code in the same notebook/file.

Loads the weights saved by dqn_pendulum.py and runs several fully-greedy
(epsilon = 0) episodes to check the final swing-up + balance behavior.

Run:
    python evaluate.py                     # prints returns, no rendering
    python evaluate.py --render            # opens a window, human-viewable (needs a local display)
    python evaluate.py --episodes 5 --render
    python evaluate.py --animate           # saves pendulum.gif you can view inline (e.g. on Colab)
    python evaluate.py --animate --gif-path my_run.gif --fps 30
"""

import argparse

import numpy as np
import torch
import torch.nn as nn
import gymnasium as gym
import matplotlib.pyplot as plt
import matplotlib.animation as animation

# --- Duplicated from dqn_pendulum.py so this file has no cross-file import ---
ENV_NAME = "Pendulum-v1"
N_ACTIONS = 21
TORQUE_MIN, TORQUE_MAX = -2.0, 2.0
MAX_STEPS_PER_EPISODE = 200
TORQUE_VALUES = np.linspace(TORQUE_MIN, TORQUE_MAX, N_ACTIONS)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def action_index_to_torque(action_index: int) -> np.ndarray:
    """Map a discrete action index to the continuous torque Pendulum-v1 expects."""
    return np.array([TORQUE_VALUES[action_index]], dtype=np.float32)


class QNetwork(nn.Module):
    """Must match the architecture used in dqn_pendulum.py exactly, or load_state_dict will fail."""

    def __init__(self, state_dim: int, n_actions: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, n_actions),
        )

    def forward(self, x):
        return self.net(x)


# --- End duplicated section ---

DEFAULT_WEIGHTS_PATH = "dqn_pendulum_weights.pt"


def load_agent(weights_path: str, state_dim: int, n_actions: int) -> QNetwork:
    q_net = QNetwork(state_dim, n_actions).to(DEVICE)
    q_net.load_state_dict(torch.load(weights_path, map_location=DEVICE))
    q_net.eval()
    return q_net


def run_episode(env, q_net, seed=None) -> float:
    state, _ = env.reset(seed=seed)
    episode_return = 0.0

    for _ in range(MAX_STEPS_PER_EPISODE):
        with torch.no_grad():
            state_t = torch.tensor(state, dtype=torch.float32, device=DEVICE).unsqueeze(0)
            action_index = int(torch.argmax(q_net(state_t), dim=1).item())

        torque = action_index_to_torque(action_index)
        state, reward, terminated, truncated, _ = env.step(torque)
        episode_return += reward

        if terminated or truncated:
            break

    return episode_return


def run_episode_with_frames(env, q_net, seed=None):
    """Same as run_episode, but also collects an rgb_array frame at every step.

    Requires `env` to have been created with render_mode="rgb_array".
    """
    state, _ = env.reset(seed=seed)
    episode_return = 0.0
    frames = [env.render()]  # capture the initial frame too, before any action

    for _ in range(MAX_STEPS_PER_EPISODE):
        with torch.no_grad():
            state_t = torch.tensor(state, dtype=torch.float32, device=DEVICE).unsqueeze(0)
            action_index = int(torch.argmax(q_net(state_t), dim=1).item())

        torque = action_index_to_torque(action_index)
        state, reward, terminated, truncated, _ = env.step(torque)
        episode_return += reward
        frames.append(env.render())

        if terminated or truncated:
            break

    return episode_return, frames


def save_animation(frames, save_path="pendulum.gif", fps=30):
    """Save a list of rgb_array frames as a GIF using matplotlib + Pillow (no ffmpeg needed).

    View it in a Colab cell with:
        from IPython.display import Image
        Image(filename="pendulum.gif")
    """
    fig, ax = plt.subplots(figsize=(4, 4))
    ax.axis("off")
    img = ax.imshow(frames[0])
    fig.tight_layout(pad=0)

    def update(frame_idx):
        img.set_data(frames[frame_idx])
        return [img]

    anim = animation.FuncAnimation(fig, update, frames=len(frames), interval=1000 / fps, blit=True)
    anim.save(save_path, writer=animation.PillowWriter(fps=fps))
    plt.close(fig)
    print(f"Saved animation to {save_path} ({len(frames)} frames)")


def main():
    parser = argparse.ArgumentParser(description="Evaluate a trained DQN pendulum policy.")
    parser.add_argument("--weights", type=str, default=DEFAULT_WEIGHTS_PATH,
                         help="Path to saved Q-network weights (.pt)")
    parser.add_argument("--episodes", type=int, default=10,
                         help="Number of greedy evaluation episodes")
    parser.add_argument("--render", action="store_true",
                         help="Open a render window to watch the policy (needs a local display)")
    parser.add_argument("--animate", action="store_true",
                         help="Save one greedy episode as a GIF (works headless, e.g. on Colab)")
    parser.add_argument("--gif-path", type=str, default="pendulum.gif",
                         help="Output path for the GIF when --animate is used")
    parser.add_argument("--fps", type=int, default=30,
                         help="Frames per second for the saved GIF")
    parser.add_argument("--seed", type=int, default=1000,
                         help="Base seed; each eval episode uses seed + episode_index")
    # parse_known_args (not parse_args) so this survives being run directly in a
    # Jupyter/Colab cell, which injects its own "-f /path/to/kernel.json" argument
    # that argparse would otherwise reject as unrecognized.
    args, _unknown = parser.parse_known_args()

    q_net = None  # loaded once state_dim is known below

    if args.animate:
        # rgb_array gives us frames to save, without needing a display (unlike "human")
        anim_env = gym.make(ENV_NAME, render_mode="rgb_array")
        state_dim = anim_env.observation_space.shape[0]
        q_net = load_agent(args.weights, state_dim, N_ACTIONS)

        ep_return, frames = run_episode_with_frames(anim_env, q_net, seed=args.seed)
        anim_env.close()
        print(f"Animated episode return: {ep_return:.1f}")
        save_animation(frames, save_path=args.gif_path, fps=args.fps)
        print(
            "\nView it inline in a notebook with:\n"
            "    from IPython.display import Image\n"
            f"    Image(filename='{args.gif_path}')"
        )
        return  # animation is a standalone action; skip the numeric-only eval loop below

    render_mode = "human" if args.render else None
    env = gym.make(ENV_NAME, render_mode=render_mode)
    state_dim = env.observation_space.shape[0]

    q_net = load_agent(args.weights, state_dim, N_ACTIONS)

    returns = []
    for ep in range(1, args.episodes + 1):
        ep_return = run_episode(env, q_net, seed=args.seed + ep)
        returns.append(ep_return)
        print(f"Eval episode {ep:2d} | return {ep_return:8.1f}")

    env.close()

    returns = np.array(returns)
    print(
        f"\nOver {args.episodes} greedy episodes: "
        f"mean {returns.mean():.1f} | std {returns.std():.1f} | "
        f"best {returns.max():.1f} | worst {returns.min():.1f}"
    )


if __name__ == "__main__":
    main()

Eval episode  1 | return   -129.4
Eval episode  2 | return   -126.8
Eval episode  3 | return   -117.0
Eval episode  4 | return   -344.1
Eval episode  5 | return   -248.7
Eval episode  6 | return   -126.4
Eval episode  7 | return   -243.8
Eval episode  8 | return   -247.3
Eval episode  9 | return   -232.5
Eval episode 10 | return   -234.3

Over 10 greedy episodes: mean -205.0 | std 72.0 | best -117.0 | worst -344.1


In [9]:
# Run this in a new cell, after the evaluate.py cell has executed
anim_env = gym.make(ENV_NAME, render_mode="rgb_array")
state_dim = anim_env.observation_space.shape[0]
q_net = load_agent("dqn_pendulum_weights.pt", state_dim, N_ACTIONS)

ep_return, frames = run_episode_with_frames(anim_env, q_net, seed=1000)
anim_env.close()
print(f"Animated episode return: {ep_return:.1f}")

save_animation(frames, save_path="pendulum.gif", fps=30)

Animated episode return: -2.7
Saved animation to pendulum.gif (201 frames)


In [11]:
from google.colab import files
files.download('pendulum.gif')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')